# 🔬 DocuAgent: construcción mínima y disección de comportamiento

Dos objetivos en un notebook:

1. **Construir DocuAgent v0**: un agente ReAct (LangGraph) que analiza las configuraciones RAG de tu `rag_memory`. Principio de diseño no negociable: *el agente redacta, las herramientas calculan* — todo número sale de una tool determinista.
2. **Estudiarlo pormenorizadamente**: no leer el código, sino observar su conducta — qué decide, en qué orden llama herramientas, cuánto cuesta, si es estable, si se recupera de fallos y si se inventa números.

Secciones: 1) tools · 2) agente · 3) anatomía del grafo · 4) traza paso a paso · 5) contador de llamadas/latencias · 6) viaje en el tiempo (checkpoints) · 7) test de estabilidad · 8) inyección de fallos · 9) auditor de grounding.

## 1 · Las herramientas: donde vive la inteligencia protectora
Cada tool devuelve **agregados calculados con numpy**, nunca el JSON crudo. Los docstrings importan más de lo que parece: **son el prompt de la herramienta** — el modelo decide cuándo llamarla leyéndolos. Escríbelos como si le explicaras a un analista junior qué hace y cuándo usarla.

In [ ]:
from langchain_core.tools import tool

from rag_lab.memory import Memoria
from rag_lab.analisis import observatorio as obs

mem = Memoria()
print(obs.panorama(mem))

# ERA: fecha del último reset de memoria — evita mezclar escalas de métricas.
DESDE = None   # p. ej. "2026-07-05"


@tool
def listar_documentos() -> str:
    """Lista los documentos estudiados con su perfil. Úsala SIEMPRE primero
    para saber qué evidencia existe antes de afirmar nada."""
    filas = []
    for s in mem._leer().get("sesiones", []):
        p = s.get("perfil") or {}
        filas.append(f"- {s.get('pdf_nombre')} | perfil={p.get('descriptor', '?')} "
                     f"| preguntas={s.get('n_preguntas')}")
    return "\n".join(filas) or "No hay documentos estudiados."


@tool
def rendimiento_de_etiqueta(banco: str, etiqueta: str) -> str:
    """Rendimiento agregado de UNA config (media + IC bootstrap sobre todas sus
    mediciones históricas). Todo calculado por el observatorio, nada estimado."""
    r = obs.rendimiento(mem, banco, etiqueta, desde=DESDE)
    if not r.get("disponible"):
        return f"Sin mediciones de '{etiqueta}' en el banco {banco}."
    return (f"{etiqueta} [{banco}]: n={r['n']} | media={r['media']} "
            f"| IC95=[{r['ic_bajo']}, {r['ic_alto']}]")


@tool
def comparar_etiquetas(banco: str, etiqueta_a: str, etiqueta_b: str) -> str:
    """Comparación PAREADA de dos configs con desglose por documento. Única
    fuente válida para afirmar que una config 'gana': si dice CONCLUYENTE=False,
    es un empate y debe decirse así. El desglose revela victorias condicionales
    ('A gana en documentos densos, pierde en los léxicos')."""
    d = obs.duelo(mem, banco, etiqueta_a, etiqueta_b, desde=DESDE)
    if not d.get("disponible"):
        return f"Evidencia insuficiente: {d.get('motivo')}"
    g = d["global"]
    por_doc = "; ".join(f"{doc}: {v['dif_media']:+}" for doc, v in d["por_documento"].items())
    return (f"{etiqueta_a} vs {etiqueta_b} [{banco}] n={d['n']}: "
            f"dif={g['dif_media']:+} IC=[{g['ic_bajo']}, {g['ic_alto']}] "
            f"CONCLUYENTE={g['concluyente']} | por documento: {por_doc}")


@tool
def victorias_por_perfil(banco: str) -> str:
    """Qué config gana en qué TIPO de documento (tabla ganador x perfil).
    Úsala para preguntas de patrones: '¿qué funciona en documentos densos?'."""
    t = obs.victorias_por_perfil(mem, banco, desde=DESDE)
    return str(t) if t else f"Sin veredictos del banco {banco}."


TOOLS = [listar_documentos, rendimiento_de_etiqueta, comparar_etiquetas,
         victorias_por_perfil]
print("Tools:", [t.name for t in TOOLS])

Fíjate en los guardarraíles *dentro* de las tools: `comparar_etiquetas` se niega con <3 pares (el agente no puede sacar conclusiones de evidencia que la tool no le da) y declara explícitamente que es "la única fuente válida" para afirmar victorias — instrucción de comportamiento incrustada en la herramienta, no en el prompt del sistema. Es más robusto: viaja con la tool.

## 2 · El agente
ReAct prefabricado de LangGraph: bucle razonar → llamar tool → observar → repetir, con checkpointer para poder viajar por su historia. El system prompt fija el contrato de honestidad.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

from rag_lab.fabricas import construir_llm_juez

SISTEMA = (
    "Eres DocuAgent, analista de experimentos RAG. Reglas de honestidad:\n"
    "1. TODO número de tu respuesta debe proceder literalmente de una herramienta. "
    "Si no tienes el dato, dilo; jamás lo estimes.\n"
    "2. Solo afirmas que una configuración supera a otra si comparar_etiquetas "
    "devuelve CONCLUYENTE=True. Si no, lo llamas empate.\n"
    "3. No inventes explicaciones causales (por qué gana algo); como mucho, "
    "hipótesis marcadas explícitamente como hipótesis.\n"
    "4. Menciona siempre el tamaño de la evidencia (n) junto a cada conclusión."
)

checkpointer = MemorySaver()
agente = create_react_agent(construir_llm_juez(), TOOLS,
                            prompt=SISTEMA, checkpointer=checkpointer)
print("DocuAgent v0 listo.")

## 3 · Anatomía del grafo
Antes de verlo correr, mira su esqueleto: un ReAct son dos nodos (`agent` y `tools`) y una arista condicional. Toda la complejidad conductual emerge de ese bucle mínimo.

In [ ]:
def dibujar(grafo) -> None:
    """ASCII si grandalf está instalado; si no, Mermaid (sin dependencias).
    Pega el Mermaid en https://mermaid.live para verlo dibujado."""
    try:
        print(grafo.get_graph().draw_ascii())
    except ImportError:
        print("(grandalf no instalado: `pip install grandalf` para ASCII; "
              "mientras, Mermaid:)\n")
        print(grafo.get_graph().draw_mermaid())


dibujar(agente)

## 4 · Traza paso a paso: ver al agente pensar
`stream_mode="updates"` emite el estado tras cada nodo: verás cada decisión del LLM (¿llamo tool o respondo?), cada llamada con sus argumentos y cada observación. Esta celda es la lupa principal del notebook.

In [ ]:
import json as _json

def trazar(pregunta: str, thread: str = "traza-1"):
    config = {"configurable": {"thread_id": thread}}
    print(f"PREGUNTA: {pregunta}\n" + "=" * 70)
    for evento in agente.stream({"messages": [("user", pregunta)]}, config,
                                stream_mode="updates"):
        for nodo, cambio in evento.items():
            for msg in cambio.get("messages", []):
                tipo = type(msg).__name__
                if getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        print(f"[{nodo}] -> LLAMA {tc['name']}({_json.dumps(tc['args'], ensure_ascii=False)})")
                elif tipo == "ToolMessage":
                    contenido = str(msg.content)
                    print(f"[{nodo}] <- {msg.name}: {contenido[:180]}"
                          + ("..." if len(contenido) > 180 else ""))
                elif getattr(msg, "content", None):
                    print(f"[{nodo}] RESPUESTA FINAL:\n{msg.content}")
    return config

config_1 = trazar("¿Qué embedding rinde mejor según la evidencia, bge-m3 o voyage-3-lite? "
                  "¿Es una diferencia real?")

**Qué mirar en la traza**: ¿empezó por `listar_documentos` (contexto) o fue directo a comparar (eficiente pero ciego al alcance de la evidencia)? ¿Llamó `comparar_etiquetas` antes de afirmar superioridad, como exige la regla 2? ¿La respuesta respeta el CONCLUYENTE de la tool o lo endulza?

## 5 · Contador: cuánto cuesta pensar
Un callback casero cuenta llamadas al LLM, tokens y cronometra cada tool. Es tu cuentakilómetros: si una pregunta simple dispara 8 llamadas al LLM, el diseño de tools está obligando a iterar de más.

In [ ]:
import time
from langchain_core.callbacks import BaseCallbackHandler

class Contador(BaseCallbackHandler):
    def __init__(self):
        self.llm_calls = 0; self.tokens_in = 0; self.tokens_out = 0
        self.tools = []; self._t0 = {}
    def on_chat_model_start(self, serialized, messages, **kw):
        self.llm_calls += 1
    def on_llm_end(self, response, **kw):
        try:
            uso = response.generations[0][0].message.usage_metadata or {}
            self.tokens_in += uso.get("input_tokens", 0)
            self.tokens_out += uso.get("output_tokens", 0)
        except Exception:
            pass
    def on_tool_start(self, serialized, input_str, *, run_id, **kw):
        self._t0[run_id] = time.perf_counter()
        self.tools.append({"tool": (serialized or {}).get("name", "?"), "s": None})
    def on_tool_end(self, output, *, run_id, **kw):
        if run_id in self._t0 and self.tools:
            self.tools[-1]["s"] = round(time.perf_counter() - self._t0.pop(run_id), 3)

cont = Contador()
config = {"configurable": {"thread_id": "contado-1"}, "callbacks": [cont]}
res = agente.invoke({"messages": [("user",
    "Resume la evidencia disponible del banco de embeddings en 3 frases.")]}, config)

print(f"Llamadas LLM: {cont.llm_calls} | tokens in/out: {cont.tokens_in}/{cont.tokens_out}")
for t in cont.tools:
    print(f"  tool {t['tool']}: {t['s']}s")
print("\n" + res["messages"][-1].content)

## 6 · Viaje en el tiempo: la historia interna del agente
El checkpointer guardó un checkpoint por paso. Puedes listar la historia, ver el estado en cualquier punto intermedio y **re-ejecutar desde un checkpoint antiguo** — la herramienta definitiva para responder "¿y si en este punto hubiera pasado otra cosa?".

In [ ]:
historia = list(agente.get_state_history(config_1))
print(f"{len(historia)} checkpoints del hilo 'traza-1' (del más reciente al inicial):\n")
for h in historia:
    n_msgs = len(h.values.get("messages", []))
    print(f"  checkpoint {h.config['configurable']['checkpoint_id'][:12]}... "
          f"| siguiente nodo: {h.next or '(fin)'} | mensajes acumulados: {n_msgs}")

# Re-ejecutar desde un punto intermedio (fork de la historia):
if len(historia) >= 3:
    punto = historia[2]           # dos pasos antes del final
    print("\n--- Fork desde el checkpoint intermedio: el agente repite desde ahí ---")
    for ev in agente.stream(None, punto.config, stream_mode="updates"):
        for nodo, cambio in ev.items():
            for m in cambio.get("messages", []):
                if getattr(m, "content", None) and not getattr(m, "tool_calls", None):
                    print(f"[{nodo}] {str(m.content)[:200]}")

## 7 · Test de estabilidad: ¿es el mismo analista dos veces seguidas?
Tu laboratorio es determinista de punta a punta; el agente no tiene por qué serlo (aun a temperatura 0, el orden de tool-calls puede variar). Misma pregunta N veces, comparamos la *secuencia de herramientas* y la conclusión. Si la ruta baila pero la conclusión aguanta, bien; si la conclusión baila, el agente no es un instrumento fiable para esa pregunta.

In [ ]:
PREGUNTA_TEST = ("¿Hay diferencia concluyente entre granite-311m-r2 y granite-97m-r2 "
                 "en el banco de embeddings?")
rutas, respuestas = [], []
for i in range(3):
    cfg = {"configurable": {"thread_id": f"estabilidad-{i}"}}
    r = agente.invoke({"messages": [("user", PREGUNTA_TEST)]}, cfg)
    ruta = [tc["name"] for m in r["messages"]
            for tc in (getattr(m, "tool_calls", None) or [])]
    rutas.append(tuple(ruta))
    respuestas.append(r["messages"][-1].content)
    print(f"pasada {i+1}: ruta = {' -> '.join(ruta) or '(sin tools)'}")

print(f"\nRutas idénticas: {len(set(rutas)) == 1}")
concluyentes = ['concluyente' in r.lower() or 'empate' in r.lower() for r in respuestas]
print(f"Las 3 respuestas hablan de concluyencia/empate: {all(concluyentes)}")
print("\nÚltima respuesta:\n", respuestas[-1][:500])

## 8 · Inyección de fallos: ¿cómo digiere un error?
Un buen agente trata el error de una tool como observación (lo cuenta, prueba otra vía); uno malo lo ignora o alucina el dato que no obtuvo. Rompemos `rendimiento_de_etiqueta` a propósito y observamos.

In [ ]:
from langchain_core.tools import tool as _tool

@_tool
def rendimiento_de_etiqueta_rota(banco: str, etiqueta: str) -> str:
    """Rendimiento agregado de UNA configuración concreta (media e IC)."""
    raise RuntimeError("memoria corrupta: colección de experimentos ilegible")

agente_fragil = create_react_agent(
    construir_llm_juez(),
    [listar_documentos, victorias_por_perfil, rendimiento_de_etiqueta_rota, comparar_etiquetas],
    prompt=SISTEMA, checkpointer=MemorySaver())

cfg = {"configurable": {"thread_id": "fallo-1"}}
r = agente_fragil.invoke({"messages": [("user",
    "¿Cuál es el rendimiento medio de bge-m3 en embeddings?")]}, cfg)
for m in r["messages"]:
    if type(m).__name__ == "ToolMessage":
        print(f"[tool {m.name}] -> {str(m.content)[:120]}")
print("\nRESPUESTA ANTE EL FALLO:\n", r["messages"][-1].content)

## 9 · Auditor de grounding: ¿se inventó algún número?
El contrato dice que todo número debe venir de una tool. Se comprueba **programáticamente**: se extraen los números de la respuesta final y se verifica que cada uno aparece en alguna salida de herramienta de esa conversación. Es el 'anclaje' del agente — el mismo espíritu que el anclaje humano de la métrica.

In [ ]:
import re

def auditar_grounding(resultado) -> None:
    salidas_tools = " ".join(str(m.content) for m in resultado["messages"]
                             if type(m).__name__ == "ToolMessage")
    final = resultado["messages"][-1].content
    numeros = set(re.findall(r"-?\d+(?:[.,]\d+)?", final))
    numeros = {n for n in numeros if len(n.replace('-', '').replace('.', '')) > 1
               or float(n.replace(',', '.')) > 9}   # ignora enumeraciones 1,2,3
    sin_fuente = [n for n in numeros
                  if n not in salidas_tools and n.replace(",", ".") not in salidas_tools]
    print(f"Números en la respuesta: {sorted(numeros) or '(ninguno)'}")
    if sin_fuente:
        print(f"⚠ SIN FUENTE EN LAS TOOLS: {sorted(sin_fuente)} — posible invención.")
    else:
        print("✓ Grounding OK: todo número procede de una herramienta.")

cfg = {"configurable": {"thread_id": "grounding-1"}}
r = agente.invoke({"messages": [("user",
    "Compara bge-m3 con voyage-3-lite en embeddings y dame cifras.")]}, cfg)
print(r["messages"][-1].content, "\n" + "=" * 60)
auditar_grounding(r)

## 10 · Guía de lectura y siguientes pasos

Con las nueve sondas anteriores tienes el perfil conductual completo de DocuAgent:

| Sonda | Pregunta que responde |
|---|---|
| Traza (§4) | ¿Qué decide y en qué orden? ¿Respeta el contrato? |
| Contador (§5) | ¿Cuánto cuesta cada pregunta? ¿Itera de más? |
| Time travel (§6) | ¿Qué había en su cabeza en cada paso? |
| Estabilidad (§7) | ¿Es un instrumento o una tómbola? |
| Fallos (§8) | ¿Degrada con honestidad o alucina? |
| Grounding (§9) | ¿Inventó números? |

**Siguientes pasos naturales**: añadir tools de filtrado por *era de métricas* (que el agente no mezcle escalas pre/post recalibración — la trampa nº 1 de tu memoria); una tool de perfil de documento para que condicione sus análisis ("en documentos densos..."); y cuando el agente madure, pasarlo por el mismo tratamiento que tu métrica: un anclaje humano de sus análisis (¿sus conclusiones coinciden con las tuyas leyendo los mismos datos?).

**Aviso de coste**: cada pregunta al agente son 2-5 llamadas a tu `modelo_juez` (Opus). Para sesiones largas de estudio, cambia el modelo del agente por uno más barato — el comportamiento del *bucle* se estudia igual.